<!--nav--> [🗺 Learning path](README.md) · **26/39** · ◀ [Serving Internals Visualized](./Serving_Internals_Visualized_D3.ipynb) · [Benchmark & Capacity Planning](./Serving_Benchmark_Capacity_Planning.ipynb) ▶

# Reading the Logs: vLLM Observability, Line by Line

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/Serving_Logs_Observability.ipynb)

Every optimization in notebooks 21–25 is invisible in production unless you can **read the logs**.
A running engine tells you — in two boring-looking lines and one metrics endpoint — whether your KV
cache is starving, whether prefix caching is working, whether requests are being preempted, and
whether you should be adding GPUs or just changing a flag.

This notebook teaches you to read all of it, **field by field**, and then turn it into pictures.

| Part | What you'll learn |
|---|---|
| **1** | The **startup log**, line by line — the 4 lines that decide your capacity before traffic arrives |
| **2** | The **periodic stats line** — 6 fields, what each one means, and what "healthy" looks like |
| **3** | **Prometheus `/metrics`** — counters vs gauges vs histograms, and how to compute p50/p95/p99 from buckets *by hand* |
| **4** | **Derived signals** — the 7 numbers worth alerting on (and the ratios that expose the real bottleneck) |
| **5** | **The debugging playbook** — 6 real symptoms, each with a log excerpt to diagnose |
| **6** | **An animated incident** — watch a traffic ramp saturate KV, trigger preemption, and blow the SLO |
| **7** | Wiring it up: Grafana, alert thresholds, and what to log per-request |

**Runs on:** any CPU (free Colab, your laptop). Realistic log/metrics samples are embedded, so every
parser runs offline. **Optional:** a GPU cell at the end regenerates the same logs from a live vLLM
server so you can point these parsers at the real thing.

> ⚠️ **Log formats drift between vLLM versions.** The *fields* below have been stable for a long
> time, but exact wording and line prefixes change. That's why every parser here is written
> **tolerantly** (named regex groups, optional fields) instead of splitting on fixed columns — and
> why Part 3's `/metrics` is the interface you should build dashboards on. Metric *names* are far
> more stable than log *text*.

In [ ]:
# No GPU, no network needed. Everything in Parts 1-6 runs on these embedded samples.
import re, json, math, statistics
from collections import defaultdict
print("ready - Parts 1-6 are pure Python over embedded, representative samples")

## Part 1 · The startup log: your capacity is decided here

When `vllm serve` boots, it prints a wall of INFO lines. Most are noise. **Four of them determine
how many users you can serve** — and people routinely deploy without ever reading them.

Here is a representative startup log (Qwen2.5-0.5B on a 16 GB T4, `--max-model-len 2048`):

In [ ]:
STARTUP_LOG = r'''
INFO 08-08 07:12:03 [__init__.py:239] Automatically detected platform cuda.
INFO 08-08 07:12:11 [api_server.py:1043] vLLM API server version 0.10.0
INFO 08-08 07:12:11 [cli_args.py:261] non-default args: {'model': 'Qwen/Qwen2.5-0.5B-Instruct', 'dtype': 'half', 'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'max_num_seqs': 32}
INFO 08-08 07:12:14 [config.py:1604] Using max model len 2048
INFO 08-08 07:12:20 [core.py:572] Waiting for init message from front-end.
INFO 08-08 07:12:21 [gpu_model_runner.py:1843] Starting to load model Qwen/Qwen2.5-0.5B-Instruct...
INFO 08-08 07:12:24 [default_loader.py:262] Loading weights took 2.41 seconds
INFO 08-08 07:12:25 [gpu_model_runner.py:1892] Model loading took 0.9270 GiB and 3.612345 seconds
INFO 08-08 07:12:41 [backends.py:530] Using cache directory: /root/.cache/vllm/torch_compile_cache/a1b2c3
INFO 08-08 07:12:55 [monitor.py:34] torch.compile takes 12.05 s in total
INFO 08-08 07:12:56 [gpu_worker.py:255] Available KV cache memory: 11.34 GiB
INFO 08-08 07:12:57 [kv_cache_utils.py:833] GPU KV cache size: 371,264 tokens
INFO 08-08 07:12:57 [kv_cache_utils.py:837] Maximum concurrency for 2,048 tokens per request: 181.28x
INFO 08-08 07:13:10 [gpu_model_runner.py:2485] Graph capturing finished in 13 secs, took 0.51 GiB
INFO 08-08 07:13:11 [core.py:193] init engine (profile, create kv cache, warmup model) took 46.31 seconds
INFO 08-08 07:13:12 [api_server.py:1090] Starting vLLM API server on http://0.0.0.0:8000
'''.strip()

# A tolerant extractor: each signal is a named pattern, missing ones just report None.
STARTUP_SIGNALS = {
    "weights_gib":     r"Model loading took ([\d.]+) GiB",
    "kv_gib":          r"Available KV cache memory: ([\d.]+) GiB",
    "kv_tokens":       r"GPU KV cache size: ([\d,]+) tokens",
    "max_concurrency": r"Maximum concurrency for ([\d,]+) tokens per request: ([\d.]+)x",
    "graph_gib":       r"Graph capturing finished in \d+ secs, took ([\d.]+) GiB",
    "init_seconds":    r"init engine .*? took ([\d.]+) seconds",
}

def parse_startup(log):
    out = {}
    for key, pat in STARTUP_SIGNALS.items():
        m = re.search(pat, log)
        out[key] = m.groups() if (m and len(m.groups()) > 1) else (m.group(1) if m else None)
    return out

s = parse_startup(STARTUP_LOG)
for k, v in s.items():
    print(f"{k:<18} {v}")

### The four lines that matter — and what each one is telling you

**1. `Model loading took 0.9270 GiB`** — your weights. Compare it to the checkpoint size: if it's
4× smaller than you expected, your quantization (notebook 23) is working. If a 7B model reports
~14 GiB, you're running fp16 and *paying for it in KV cache you no longer have*.

**2. `Available KV cache memory: 11.34 GiB`** — what's left after weights + activations +
CUDA graphs, times `--gpu-memory-utilization`. This is the number every optimization is fighting for.

```
total VRAM × gpu_memory_utilization  −  weights  −  activation peak  −  CUDA graphs  =  KV pool
     16 GB  ×        0.85            −   0.93    −      ~1.1         −     0.51      ≈  11.3 GB
```

**3. `GPU KV cache size: 371,264 tokens`** — the pool expressed in tokens, which is the unit you
should think in. Total tokens (prompt + generated) across *all concurrent requests* must fit here.

**4. `Maximum concurrency for 2,048 tokens per request: 181.28x`** — vLLM doing the division for
you: `371,264 / 2,048 ≈ 181` full-length requests at once. **This is your theoretical concurrency
ceiling.** If it says `2.4x`, no amount of `--max-num-seqs 256` will help you — you're KV-bound and
the fixes are: shrink `--max-model-len`, quantize weights, or quantize the KV cache.

Let's turn the startup log into a capacity statement automatically:

In [ ]:
def capacity_report(log, gpu_total_gib=16.0, gpu_util=0.85):
    s = parse_startup(log)
    w  = float(s["weights_gib"]) if s["weights_gib"] else float("nan")
    kv = float(s["kv_gib"])      if s["kv_gib"]      else float("nan")
    toks = int(s["kv_tokens"].replace(",", ""))         if s["kv_tokens"] else 0
    ctx, conc = (int(s["max_concurrency"][0].replace(",", "")),
                 float(s["max_concurrency"][1])) if s["max_concurrency"] else (0, 0.0)
    budget = gpu_total_gib * gpu_util
    overhead = budget - w - kv - (float(s["graph_gib"]) or 0)

    print(f"VRAM budget ({gpu_util:.0%} of {gpu_total_gib:.0f} GiB) : {budget:6.2f} GiB")
    print(f"  weights                              : {w:6.2f} GiB  ({w/budget:5.1%})")
    print(f"  CUDA graphs                          : {float(s['graph_gib']):6.2f} GiB")
    print(f"  activation peak + misc (derived)     : {overhead:6.2f} GiB")
    print(f"  >> KV CACHE POOL                     : {kv:6.2f} GiB  ({kv/budget:5.1%})")
    print()
    print(f"KV pool holds {toks:,} tokens  =>  {conc:.0f} concurrent requests at {ctx:,} tokens each")
    print(f"Rules of thumb from this one report:")
    print(f"  - at 512-token conversations you could hold ~{toks//512} of them")
    print(f"  -每 request costs ~{kv*1024/ctx:.2f} MiB of KV at full length".replace("每 ", "each "))
    if conc < 8:
        print("  ⚠ concurrency ceiling is LOW - you are KV-bound. Fix: lower --max-model-len,")
        print("    quantize weights (nb 23), or use --kv-cache-dtype fp8.")
    else:
        print(f"  ✓ headroom is fine; --max-num-seqs (scheduler cap) will bind before KV does")

capacity_report(STARTUP_LOG)

## Part 2 · The periodic stats line — the heartbeat

Every few seconds a running engine prints one line. **This single line is 80% of production
debugging.** Here it is, annotated:

```
INFO 08-08 07:15:23 [loggers.py:122] Engine 000: Avg prompt throughput: 1043.2 tokens/s, Avg generation throughput: 412.7 tokens/s, Running: 24 reqs, Waiting: 0 reqs, GPU KV cache usage: 18.4%, Prefix cache hit rate: 31.2%
                                                 └──────── PREFILL rate ────────┘  └──────── DECODE rate ────────┘  └── in flight ─┘  └─ queued ─┘  └── KV pool used ──┘  └─ prefix reuse ─┘
```

| Field | Means | Healthy | Alarm |
|---|---|---|---|
| **Avg prompt throughput** | prefill tokens/s — how fast prompts are being ingested | spiky (bursts with arrivals) | pinned high *and* decode near zero → prefill is starving decode |
| **Avg generation throughput** | decode tokens/s — what users experience as streaming | steady, scales with Running | drops while Running is flat → contention or long contexts |
| **Running** | requests in the current batch | near `--max-num-seqs` under load | always 1–2 under load → scheduler can't admit; look at KV usage |
| **Waiting** | requests queued, *not yet admitted* | 0 most of the time | persistently > 0 → you are at capacity; queue time is now in every user's TTFT |
| **GPU KV cache usage** | fraction of the token pool in use | 30–80% | > 90% sustained → preemption imminent; ~0% with Waiting > 0 → something else is the limit |
| **Prefix cache hit rate** | share of prompt blocks served from cache | high for chat/RAG (30–90%) | ~0% on repeated system prompts → your prompt layout defeats caching (nb 22) |

Now the parser. Note the deliberate tolerance: every field is optional, so a version that renames or
drops one doesn't crash your pipeline.

In [ ]:
STATS_RE = re.compile(
    r"Avg prompt throughput:\s*(?P<prompt_tps>[\d.]+)\s*tokens/s"
    r".*?Avg generation throughput:\s*(?P<gen_tps>[\d.]+)\s*tokens/s"
    r".*?Running:\s*(?P<running>\d+)\s*reqs"
    r".*?Waiting:\s*(?P<waiting>\d+)\s*reqs"
    r"(?:.*?GPU KV cache usage:\s*(?P<kv_pct>[\d.]+)\s*%)?"
    r"(?:.*?Prefix cache hit rate:\s*(?P<prefix_pct>[\d.]+)\s*%)?",
    re.S)
TS_RE = re.compile(r"(?P<mon>\d{2})-(?P<day>\d{2})\s+(?P<h>\d{2}):(?P<m>\d{2}):(?P<s>\d{2})")

def parse_stats_line(line):
    m = STATS_RE.search(line)
    if not m:
        return None
    d = {k: (float(v) if v is not None else None) for k, v in m.groupdict().items()}
    t = TS_RE.search(line)
    if t:
        d["t"] = int(t["h"]) * 3600 + int(t["m"]) * 60 + int(t["s"])
    return d

SAMPLE_LINES = [
 "INFO 08-08 07:15:23 [loggers.py:122] Engine 000: Avg prompt throughput: 1043.2 tokens/s, Avg generation throughput: 412.7 tokens/s, Running: 24 reqs, Waiting: 0 reqs, GPU KV cache usage: 18.4%, Prefix cache hit rate: 31.2%",
 "INFO 08-08 07:15:33 [loggers.py:122] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 588.1 tokens/s, Running: 31 reqs, Waiting: 0 reqs, GPU KV cache usage: 41.7%, Prefix cache hit rate: 48.9%",
 "INFO 08-08 07:15:43 [loggers.py:122] Engine 000: Avg prompt throughput: 2210.6 tokens/s, Avg generation throughput: too-old-format",   # deliberately broken
 "INFO 08-08 07:15:53 [loggers.py:122] Engine 000: Avg prompt throughput: 118.4 tokens/s, Avg generation throughput: 233.0 tokens/s, Running: 32 reqs, Waiting: 47 reqs, GPU KV cache usage: 96.8%",
]

for ln in SAMPLE_LINES:
    p = parse_stats_line(ln)
    print(("OK   " if p else "SKIP ") + (json.dumps(p) if p else ln[:90] + " ..."))

print("\nNote the 3rd line: unparseable, and the parser returns None instead of crashing.")
print("The 4th line has no prefix-cache field (older build) - it still parses, prefix_pct=None.")
print("\nRead the 4th line like an SRE: Waiting=47 with KV at 96.8% and decode DOWN to 233 tok/s.")
print("That is the signature of KV exhaustion -> preemption. Part 5 covers the fix.")

## Part 3 · Prometheus `/metrics` — the interface you should actually build on

Logs are for humans; `/metrics` is for machines. vLLM exposes it on the same port
(`curl localhost:8000/metrics`). Three metric *types*, and confusing them is the classic mistake:

| Type | Behavior | Example | How to use it |
|---|---|---|---|
| **Counter** | monotonically increasing; resets to 0 on restart | `vllm:generation_tokens_total` | **always take a rate**: `rate(x[1m])`. Never graph the raw value |
| **Gauge** | goes up and down; a snapshot of *now* | `vllm:num_requests_running`, `vllm:gpu_cache_usage_perc` | graph directly; alert on sustained thresholds |
| **Histogram** | cumulative bucket counts + `_sum` + `_count` | `vllm:time_to_first_token_seconds` | percentiles — but see the math below |

Here is a realistic scrape (trimmed to what matters):

In [ ]:
METRICS = r'''
# HELP vllm:num_requests_running Number of requests currently running on GPU.
# TYPE vllm:num_requests_running gauge
vllm:num_requests_running{model_name="Qwen/Qwen2.5-0.5B-Instruct"} 28.0
# TYPE vllm:num_requests_waiting gauge
vllm:num_requests_waiting{model_name="Qwen/Qwen2.5-0.5B-Instruct"} 12.0
# TYPE vllm:gpu_cache_usage_perc gauge
vllm:gpu_cache_usage_perc{model_name="Qwen/Qwen2.5-0.5B-Instruct"} 0.873
# TYPE vllm:num_preemptions_total counter
vllm:num_preemptions_total{model_name="Qwen/Qwen2.5-0.5B-Instruct"} 43.0
# TYPE vllm:prompt_tokens_total counter
vllm:prompt_tokens_total{model_name="Qwen/Qwen2.5-0.5B-Instruct"} 1284301.0
# TYPE vllm:generation_tokens_total counter
vllm:generation_tokens_total{model_name="Qwen/Qwen2.5-0.5B-Instruct"} 402118.0
# TYPE vllm:prefix_cache_queries_total counter
vllm:prefix_cache_queries_total{model_name="Qwen/Qwen2.5-0.5B-Instruct"} 1284301.0
# TYPE vllm:prefix_cache_hits_total counter
vllm:prefix_cache_hits_total{model_name="Qwen/Qwen2.5-0.5B-Instruct"} 561894.0
# TYPE vllm:request_success_total counter
vllm:request_success_total{finished_reason="stop",model_name="Qwen/Qwen2.5-0.5B-Instruct"} 3810.0
vllm:request_success_total{finished_reason="length",model_name="Qwen/Qwen2.5-0.5B-Instruct"} 292.0
# TYPE vllm:time_to_first_token_seconds histogram
vllm:time_to_first_token_seconds_bucket{le="0.05",model_name="m"} 210.0
vllm:time_to_first_token_seconds_bucket{le="0.1",model_name="m"} 980.0
vllm:time_to_first_token_seconds_bucket{le="0.25",model_name="m"} 2450.0
vllm:time_to_first_token_seconds_bucket{le="0.5",model_name="m"} 3320.0
vllm:time_to_first_token_seconds_bucket{le="1.0",model_name="m"} 3811.0
vllm:time_to_first_token_seconds_bucket{le="2.5",model_name="m"} 4032.0
vllm:time_to_first_token_seconds_bucket{le="5.0",model_name="m"} 4095.0
vllm:time_to_first_token_seconds_bucket{le="+Inf",model_name="m"} 4102.0
vllm:time_to_first_token_seconds_sum{model_name="m"} 1523.77
vllm:time_to_first_token_seconds_count{model_name="m"} 4102.0
# TYPE vllm:time_per_output_token_seconds histogram
vllm:time_per_output_token_seconds_bucket{le="0.01",model_name="m"} 1200.0
vllm:time_per_output_token_seconds_bucket{le="0.025",model_name="m"} 3100.0
vllm:time_per_output_token_seconds_bucket{le="0.05",model_name="m"} 3900.0
vllm:time_per_output_token_seconds_bucket{le="0.1",model_name="m"} 4080.0
vllm:time_per_output_token_seconds_bucket{le="+Inf",model_name="m"} 4102.0
vllm:time_per_output_token_seconds_sum{model_name="m"} 121.4
vllm:time_per_output_token_seconds_count{model_name="m"} 4102.0
'''.strip()

SAMPLE_RE = re.compile(r'^(?P<name>[a-zA-Z_:][\w:]*)(?P<labels>\{[^}]*\})?\s+(?P<value>[-\d.eE+]+)$')
LABEL_RE = re.compile(r'(\w+)="([^"]*)"')

def parse_prometheus(text):
    out = []
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        m = SAMPLE_RE.match(line)
        if not m:
            continue
        labels = dict(LABEL_RE.findall(m["labels"] or ""))
        out.append({"name": m["name"], "labels": labels, "value": float(m["value"])})
    return out

samples = parse_prometheus(METRICS)
by_name = defaultdict(list)
for s in samples:
    by_name[s["name"]].append(s)
print(f"parsed {len(samples)} samples across {len(by_name)} metric names\n")
for n in ("vllm:num_requests_running", "vllm:num_requests_waiting",
          "vllm:gpu_cache_usage_perc", "vllm:num_preemptions_total"):
    print(f"{n:<38} {by_name[n][0]['value']}")

### Computing percentiles from histogram buckets — the math nobody explains

A Prometheus histogram gives you **cumulative counts**: `le="0.25" → 2450` means *2450 requests had
TTFT ≤ 0.25s*. To get p95 you:

1. Multiply total count by 0.95 → the **rank** you're looking for.
2. Walk buckets until the cumulative count passes that rank.
3. **Linearly interpolate** inside that bucket.

Step 3 is why histogram percentiles are *estimates*: within a bucket you assume a uniform spread.
The error is bounded by the bucket width — which is why a p99 that lands in the `+Inf` bucket is
unknowable (you'll see `+Inf` returned), and why bucket boundaries are a design decision.

Here's the same algorithm Prometheus's `histogram_quantile()` uses, written out:

In [ ]:
def histogram_buckets(samples, metric):
    bs = [(float("inf") if s["labels"]["le"] == "+Inf" else float(s["labels"]["le"]), s["value"])
          for s in samples if s["name"] == metric + "_bucket"]
    return sorted(bs)

def histogram_quantile(q, buckets):
    # buckets: sorted [(le, cumulative_count)] including a +Inf bucket. Mirrors Prometheus.
    if not buckets: return float("nan")
    total = buckets[-1][1]
    if total == 0: return float("nan")
    rank = q * total
    prev_le, prev_count = 0.0, 0.0
    for le, count in buckets:
        if count >= rank:
            if math.isinf(le):
                return float("inf")                      # p lands beyond the last finite bucket
            if count == prev_count:
                return le
            frac = (rank - prev_count) / (count - prev_count)
            return prev_le + frac * (le - prev_le)       # linear interpolation inside the bucket
        prev_le, prev_count = le, count
    return buckets[-1][0]

ttft = histogram_buckets(samples, "vllm:time_to_first_token_seconds")
tpot = histogram_buckets(samples, "vllm:time_per_output_token_seconds")

print("TTFT buckets (le, cumulative):", [(f"{le}", int(c)) for le, c in ttft])
print()
print(f"{'quantile':>9}{'TTFT':>10}{'TPOT':>10}")
for q in (0.5, 0.9, 0.95, 0.99):
    print(f"{q:>9.2f}{histogram_quantile(q, ttft)*1000:>8.0f}ms{histogram_quantile(q, tpot)*1000:>8.1f}ms")

# The mean is available exactly (_sum/_count) - and it LIES compared to the tail:
s_sum = next(s["value"] for s in samples if s["name"] == "vllm:time_to_first_token_seconds_sum")
s_cnt = next(s["value"] for s in samples if s["name"] == "vllm:time_to_first_token_seconds_count")
print(f"\nmean TTFT = _sum/_count = {s_sum/s_cnt*1000:.0f}ms")
print(f"p95  TTFT                = {histogram_quantile(0.95, ttft)*1000:.0f}ms")
print("=> the average user waits ~0.4s; 1 in 20 waits ~3x longer. ALWAYS alert on percentiles.")

## Part 4 · Derived signals: the 7 numbers worth alerting on

Raw metrics are not insight. These **ratios** are — each one isolates a different failure mode:

| # | Signal | Formula | Read it as |
|---|---|---|---|
| 1 | **KV pressure** | `gpu_cache_usage_perc` | >0.9 sustained = preemption territory |
| 2 | **Queue pressure** | `num_requests_waiting / (running + waiting)` | >0.3 = you're shedding latency onto users |
| 3 | **Preemption rate** | `rate(num_preemptions_total[5m])` | anything >0 sustained = wasted compute, recomputed prefills |
| 4 | **Prefix hit rate** | `hits_total / queries_total` | low on repetitive traffic = prompt layout bug (nb 22) |
| 5 | **Decode share** | `generation_tps / (prompt_tps + generation_tps)` | low = prefill-dominated; consider chunked prefill / disaggregation |
| 6 | **Token amplification** | `generation_tokens / prompt_tokens` | tiny = you're paying prefill to emit almost nothing (RAG smell) |
| 7 | **Goodput** | requests/s meeting *both* TTFT and TPOT SLOs | the only number a product owner should see |

Let's compute all seven from a scrape, and — critically — have the code *say what it thinks*:

In [ ]:
def derived_signals(samples, stats_line=None):
    g = {s["name"]: s["value"] for s in samples if not s["name"].endswith(("_bucket", "_sum", "_count"))}
    run  = g.get("vllm:num_requests_running", 0)
    wait = g.get("vllm:num_requests_waiting", 0)
    hits = g.get("vllm:prefix_cache_hits_total", 0)
    qs   = g.get("vllm:prefix_cache_queries_total", 1)
    pt   = g.get("vllm:prompt_tokens_total", 0)
    gt   = g.get("vllm:generation_tokens_total", 0)
    out = {
        "kv_pressure":       g.get("vllm:gpu_cache_usage_perc", 0),
        "queue_pressure":    wait / max(run + wait, 1),
        "preemptions_total": g.get("vllm:num_preemptions_total", 0),
        "prefix_hit_rate":   hits / max(qs, 1),
        "token_amplification": gt / max(pt, 1),
    }
    return out

sig = derived_signals(samples)
print(f"{'signal':<22}{'value':>10}   verdict")
print("-" * 74)
VERDICTS = [
    ("kv_pressure", lambda v: ("🔴 KV nearly full - preemption risk; cut max-model-len or quantize KV"
                               if v > 0.9 else "🟡 getting full - watch it" if v > 0.75
                               else "🟢 healthy headroom")),
    ("queue_pressure", lambda v: ("🔴 heavy queueing - queue time is now inside every user's TTFT"
                                  if v > 0.3 else "🟡 some queueing" if v > 0.05 else "🟢 no backlog")),
    ("preemptions_total", lambda v: ("🔴 requests ARE being preempted - their prefill gets recomputed"
                                     if v > 0 else "🟢 none")),
    ("prefix_hit_rate", lambda v: ("🟢 prefix caching is doing real work" if v > 0.25
                                   else "🟡 low reuse - is the shared prefix really at the FRONT?")),
    ("token_amplification", lambda v: ("🟡 prefill-heavy workload (RAG/summarization shape)" if v < 0.5
                                       else "🟢 balanced prompt:generation ratio")),
]
for key, verdict in VERDICTS:
    v = sig[key]
    shown = f"{v:.3f}" if v < 100 else f"{v:.0f}"
    print(f"{key:<22}{shown:>10}   {verdict(v)}")

print("\nThis scrape's story: KV at 87%, 12 requests queued, 43 preemptions already recorded.")
print("The engine is over-subscribed. Adding --max-num-seqs would make it WORSE, not better.")

## Part 5 · The debugging playbook

Six symptoms you will actually meet. For each: the log signature, the cause, and the fix. Read the
signature first and try to diagnose before revealing the answer below it.

| # | Symptom | Log signature | Root cause | Fix |
|---|---|---|---|---|
| 1 | TTFT spikes, decode fine | `Waiting` > 0, `Running` at `--max-num-seqs`, KV moderate | scheduler slot cap, not memory | raise `--max-num-seqs` (you have KV headroom) |
| 2 | Everything slows, throughput collapses | KV ~99%, preemptions climbing | KV exhaustion → recompute loop | lower `--max-model-len`, quantize weights/KV, or add a replica |
| 3 | Streaming stutters when long prompts arrive | prompt throughput spikes, generation → ~0 | prefill starving decode | ensure chunked prefill on; lower `--max-num-batched-tokens` |
| 4 | Chatbot slower than expected | prefix hit rate ~0% despite repeated system prompt | per-request content placed *before* shared prefix | move shared text to the front of the prompt (nb 22) |
| 5 | OOM at startup, never serves | fails right after `Available KV cache memory:` or during graph capture | `gpu_memory_utilization` too high for the real footprint | lower it to 0.80–0.85, or shrink `--max-model-len` |
| 6 | Speculation made things *slower* | acceptance rate low in spec-decode metrics | bad draft model or high batch load | lower `num_speculative_tokens`, better draft, or disable under load (nb 24) |

Now diagnose three anonymous log excerpts automatically — this is the kind of triage helper worth
having in your own repo:

In [ ]:
INCIDENTS = {
 "incident A": [
  "Avg prompt throughput: 60.1 tokens/s, Avg generation throughput: 910.4 tokens/s, Running: 32 reqs, Waiting: 24 reqs, GPU KV cache usage: 34.0%, Prefix cache hit rate: 40.1%",
  "Avg prompt throughput: 55.7 tokens/s, Avg generation throughput: 921.7 tokens/s, Running: 32 reqs, Waiting: 31 reqs, GPU KV cache usage: 35.2%, Prefix cache hit rate: 41.0%"],
 "incident B": [
  "Avg prompt throughput: 210.5 tokens/s, Avg generation throughput: 305.2 tokens/s, Running: 27 reqs, Waiting: 40 reqs, GPU KV cache usage: 98.7%, Prefix cache hit rate: 22.0%",
  "Avg prompt throughput: 402.1 tokens/s, Avg generation throughput: 150.8 tokens/s, Running: 19 reqs, Waiting: 55 reqs, GPU KV cache usage: 99.1%, Prefix cache hit rate: 21.7%"],
 "incident C": [
  "Avg prompt throughput: 6120.9 tokens/s, Avg generation throughput: 42.1 tokens/s, Running: 30 reqs, Waiting: 3 reqs, GPU KV cache usage: 55.0%, Prefix cache hit rate: 2.1%",
  "Avg prompt throughput: 5890.4 tokens/s, Avg generation throughput: 38.7 tokens/s, Running: 31 reqs, Waiting: 5 reqs, GPU KV cache usage: 57.4%, Prefix cache hit rate: 1.8%"],
}

MAX_NUM_SEQS = 32   # what the server was launched with - triage needs to know the configured cap

def triage(lines, max_num_seqs=MAX_NUM_SEQS):
    ps = [p for p in (parse_stats_line(l) for l in lines) if p]
    if not ps: return ["no parseable stats lines"]
    avg = lambda k: statistics.mean(p[k] for p in ps if p.get(k) is not None)
    kv, wait, run = avg("kv_pct"), avg("waiting"), avg("running")
    gen, prompt, prefix = avg("gen_tps"), avg("prompt_tps"), avg("prefix_pct")
    findings = []
    if kv > 90:
        findings.append(f"🔴 KV EXHAUSTION (avg {kv:.0f}%) with {wait:.0f} queued -> preemption/recompute "
                        "loop. Fix: lower --max-model-len, quantize (nb23), or add a replica.")
    elif wait > 1 and run >= max_num_seqs * 0.95 and kv < 70:
        findings.append(f"🟡 SLOT-BOUND: Running pinned at the {max_num_seqs}-req cap with {wait:.0f} waiting "
                        f"but KV only {kv:.0f}% used. Fix: raise --max-num-seqs - the memory is there.")
    if prompt > 5 * max(gen, 1):
        findings.append(f"🟡 PREFILL-DOMINATED ({prompt:.0f} prefill vs {gen:.0f} decode tok/s): long prompts "
                        "are starving decode. Fix: chunked prefill / lower --max-num-batched-tokens; "
                        "at scale, prefill-decode disaggregation (nb24).")
    if prefix is not None and prefix < 10:
        findings.append(f"🟡 PREFIX CACHE COLD ({prefix:.1f}%): if your traffic repeats a system prompt or "
                        "documents, they are not at the FRONT of the prompt (nb22).")
    return findings or ["🟢 nothing alarming - engine looks balanced"]

for name, lines in INCIDENTS.items():
    print(f"\n=== {name} ===")
    for f in triage(lines):
        print("  " + f)

**Incident A** is the happy problem: memory to spare, artificially capped. One flag fixes it.
**Incident B** is the dangerous one — note decode throughput *falling* as prefill rises: preempted
requests get their prefill recomputed, so the engine does more work and delivers less.
**Incident C** is a RAG-shaped workload: enormous prefill, tiny generation, and a cold prefix cache
— the single highest-leverage fix is prompt layout, not hardware.

## Part 6 · Watch an incident unfold

Numbers in a table are hard to feel. Let's simulate a realistic incident — traffic ramps, KV fills,
preemption starts, the queue explodes, TTFT blows the SLO — and render it as a scrubable dashboard
using the same `show_d3` pattern from [notebook 25](./Serving_Internals_Visualized_D3.ipynb).

In [ ]:
# The show_d3 helper again (each notebook stays self-contained - Kaggle/Colab can't import local .py).
import uuid
from IPython.display import HTML, display

D3_URL = "https://cdn.jsdelivr.net/npm/d3@7/dist/d3.min.js"

def show_d3(js, data=None, height=420):
    div = f"viz_{uuid.uuid4().hex[:10]}"
    html = f'''
<div id="{div}" style="width:100%;max-width:920px;font-family:system-ui,sans-serif"></div>
<script>
(function() {{
  function run() {{
    const d3 = window.d3;
    const root = d3.select("#{div}");
    const data = {json.dumps(data)};
    const W = (document.getElementById("{div}").clientWidth || 880), H = {height};
    try {{ {js} }} catch (e) {{ root.append("pre").style("color","crimson").text("viz error: " + e); }}
  }}
  if (window.d3) run();
  else {{
    const s = document.createElement("script");
    s.src = "{D3_URL}";
    s.onload = run;
    s.onerror = () => document.getElementById("{div}").textContent = "Could not load D3 from the CDN.";
    document.head.appendChild(s);
  }}
}})();
</script>'''
    display(HTML(html))
print("show_d3 ready")

In [ ]:
# Simulate 180 seconds of an engine under a traffic ramp, emitting the SAME fields the real
# stats line does. This is a queueing model, not a GPU - but the failure MODE is the real one.
import random
random.seed(7)

# A long-context chat workload: generous slot cap, but big prompts -> KV binds first.
KV_TOKENS   = 371_264      # from our startup log
MAX_SEQS    = 256          # scheduler cap (deliberately high, so memory is the real limit)
AVG_PROMPT  = 3200         # long conversations, as in a chat product with history
AVG_OUTPUT  = 220
DECODE_TPS_PER_REQ = 22    # per-request decode speed when the batch is comfortable
DECODE_CEILING = 1400      # total tokens/s the GPU can sustain (the roofline from nb 21)

def simulate_incident(seconds=300):
    running, waiting, kv_used = [], [], 0
    rows, preemptions = [], 0
    # Capacity check: this GPU retires DECODE_CEILING/AVG_OUTPUT ≈ 6.4 req/s.
    # Traffic ramps GRADUALLY past that line and then falls back — the shape of a
    # real morning peak. Watch the ORDER in which the metrics break.
    def arrival_rate(t):
        if t < 40:  return 1.0                     # calm
        if t < 150: return 1.0 + (t - 40) * 0.10   # gradual ramp: 1/s -> 12/s
        return 1.0                                 # traffic returns to normal
    for t in range(seconds):
        lam = arrival_rate(t)
        arrivals = int(lam) + (1 if random.random() < lam % 1 else 0)
        for _ in range(arrivals):
            waiting.append({"prompt": max(256, int(random.gauss(AVG_PROMPT, 700))),
                            "left": max(20, int(random.gauss(AVG_OUTPUT, 90))), "born": t})
        # admit while BOTH the slot cap and the KV pool allow. kv is tracked per request.
        while waiting and len(running) < MAX_SEQS:
            r = waiting[0]
            need = r["prompt"] + 16                       # prompt KV + first block of output
            if kv_used + need > KV_TOKENS * 0.99:
                break                                     # KV-bound admission stall
            r["kv"] = need; kv_used += need
            running.append(waiting.pop(0))
        # one second of decode, shared across the batch (roofline)
        per_req = min(DECODE_TPS_PER_REQ, DECODE_CEILING / len(running)) if running else 0
        gen_tokens = 0
        for r in list(running):
            step = max(1, int(per_req))
            r["left"] -= step
            r["kv"] += step; kv_used += step               # every generated token grows the cache
            gen_tokens += step
            if r["left"] <= 0:
                kv_used -= r["kv"]; running.remove(r)      # free exactly what this request held
        # preemption: over the line -> evict the newest; its prefill must be recomputed later
        while kv_used > KV_TOKENS * 0.985 and running:
            victim = running.pop()
            kv_used -= victim["kv"]; victim["kv"] = 0
            victim["left"] = min(victim["left"] + 40, AVG_OUTPUT)   # lost work, re-queued
            waiting.insert(0, victim); preemptions += 1
        rows.append({"t": t,
                     "running": len(running), "waiting": len(waiting),
                     "kv_pct": round(100 * kv_used / KV_TOKENS, 1),
                     "gen_tps": gen_tokens,
                     "prompt_tps": AVG_PROMPT * arrivals,
                     "preemptions": preemptions,
                     # TTFT ≈ queue wait: how long the head of the queue has been waiting
                     "ttft_est": round((t - waiting[0]["born"]) if waiting else 0.05, 2)})
    return rows

incident = simulate_incident()
peak = max(incident, key=lambda r: r["waiting"])
print(f"peak queue: {peak['waiting']} waiting at t={peak['t']}s, KV {peak['kv_pct']}%, "
      f"est TTFT {peak['ttft_est']}s")
print(f"total preemptions: {incident[-1]['preemptions']}")
print("\nsample stats lines this engine would have printed:")
for r in (incident[20], incident[90], incident[150]):
    print(f"  t={r['t']:>3}s  Running: {r['running']:>2} reqs, Waiting: {r['waiting']:>3} reqs, "
          f"GPU KV cache usage: {r['kv_pct']:>5.1f}%, gen {r['gen_tps']:>4} tok/s")

In [ ]:
# The dashboard: four stacked panels + a scrubber + an automatic diagnosis at the cursor.
JS = r'''
const M = {top: 14, right: 58, bottom: 22, left: 52};
const panelH = 86, panels = [
  {key: "running",  label: "Running (batch)", color: "#42a5f5", extra: "waiting"},
  {key: "kv_pct",   label: "KV cache usage %", color: "#ef5350", max: 100},
  {key: "gen_tps",  label: "Decode tokens/s",  color: "#66bb6a"},
  {key: "ttft_est", label: "Est. TTFT (s)",    color: "#ab47bc", slo: 1.0},
];
const iw = W - M.left - M.right;
const x = d3.scaleLinear().domain(d3.extent(data, d => d.t)).range([0, iw]);

const bar = root.append("div").style("margin-bottom", "4px");
const btn = bar.append("button").text("▶ play");
const scrub = bar.append("input").attr("type", "range").attr("min", 0).attr("max", data.length - 1)
    .attr("value", 0).style("width", "260px").style("margin-left", "10px").style("vertical-align", "middle");
const readout = root.append("div").style("font", "12.5px ui-monospace,monospace")
    .style("background", "#f7f7f9").style("padding", "6px 8px").style("border-radius", "6px")
    .style("margin-bottom", "6px").style("white-space", "pre-wrap");

const svg = root.append("svg").attr("width", W).attr("height", panels.length * panelH + M.top + M.bottom);
const cursors = [];
panels.forEach((p, i) => {
  const g = svg.append("g").attr("transform", `translate(${M.left},${M.top + i * panelH})`);
  const h = panelH - 20;
  const y = d3.scaleLinear().domain([0, p.max || d3.max(data, d => d[p.key]) * 1.1 || 1]).range([h, 0]);
  g.append("g").call(d3.axisLeft(y).ticks(3)).style("font-size", "9px");
  g.append("text").attr("x", 4).attr("y", 10).style("font-size", "11px").style("font-weight", 600)
      .style("fill", p.color).text(p.label);
  g.append("path").datum(data).attr("fill", "none").attr("stroke", p.color).attr("stroke-width", 1.8)
      .attr("d", d3.line().x(d => x(d.t)).y(d => y(d[p.key])));
  if (p.extra) {   // overlay waiting queue as a faint area
    const y2 = d3.scaleLinear().domain([0, d3.max(data, d => d[p.extra])]).range([h, 0]);
    g.append("path").datum(data).attr("fill", "#ffa726").attr("opacity", 0.22)
        .attr("d", d3.area().x(d => x(d.t)).y0(h).y1(d => y2(d[p.extra])));
    g.append("text").attr("x", iw - 2).attr("y", 10).attr("text-anchor", "end")
        .style("font-size", "10px").style("fill", "#ef8f00").text("orange = waiting queue");
  }
  if (p.slo) {
    g.append("line").attr("x1", 0).attr("x2", iw).attr("y1", y(p.slo)).attr("y2", y(p.slo))
        .attr("stroke", "#d32f2f").attr("stroke-dasharray", "4 3");
    g.append("text").attr("x", iw + 4).attr("y", y(p.slo) + 3).style("font-size", "9px")
        .style("fill", "#d32f2f").text("SLO");
  }
  if (i === panels.length - 1)
    g.append("g").attr("transform", `translate(0,${h})`).call(d3.axisBottom(x).ticks(9)).style("font-size", "9px");
  cursors.push(g.append("line").attr("y1", 0).attr("y2", h).attr("stroke", "#333").attr("stroke-width", 1));
});

function diagnose(r) {
  if (r.kv_pct > 95 && r.waiting > 10) return "🔴 KV EXHAUSTED - preempting; prefill being recomputed";
  if (r.waiting > 10) return "🟠 QUEUE BUILDING - queue time is now the dominant part of TTFT";
  if (r.waiting > 0)  return "🟡 mild backlog";
  return "🟢 healthy - queue empty, KV comfortable";
}
function render(i) {
  const r = data[i];
  scrub.property("value", i);
  cursors.forEach(c => c.attr("x1", x(r.t)).attr("x2", x(r.t)));
  readout.text(
    `t=${String(r.t).padStart(3)}s │ Running: ${String(r.running).padStart(2)} │ Waiting: ${String(r.waiting).padStart(3)} │ ` +
    `KV: ${String(r.kv_pct).padStart(5)}% │ decode: ${String(r.gen_tps).padStart(4)} tok/s │ ` +
    `preempted: ${r.preemptions}\n${diagnose(r)}`);
}
let i = 0, timer = null;
btn.on("click", () => {
  if (timer) { timer.stop(); timer = null; btn.text("▶ play"); }
  else { timer = d3.interval(() => { i = (i + 1) % data.length; render(i); }, 45); btn.text("⏸ pause"); }
});
scrub.on("input", function() { i = +this.value; render(i); });
render(0);
'''
show_d3(JS, incident, height=420)

**Scrub through it and watch the causal chain — this is the whole point of the notebook:**

1. **t≈0–40s** — calm. Queue empty, KV low, TTFT ~50ms. Everything is fine and no metric moves.
2. **t≈40s** — traffic ramps. `Running` climbs to the `--max-num-seqs` cap. Still healthy.
3. **t≈60–90s** — **KV usage crosses 90%**. This is the *leading indicator*; users feel nothing yet.
   An alert here buys you minutes. An alert on TTFT buys you nothing — it fires after users suffer.
4. **t≈90s+** — admission stalls, the orange queue area explodes, preemptions begin. Note the decode
   line **falling** while the queue grows: preempted requests recompute their prefill, so the engine
   burns compute to make *negative* progress.
5. **TTFT crosses the SLO line** — the last thing to break, and the only thing most teams monitor.

**The lesson:** alert on **KV usage and queue depth** (causes), not just latency (symptom).

## Part 7 · Wiring it up for real

**PromQL you can paste into Grafana:**

```promql
# throughput (ALWAYS rate a counter)
rate(vllm:generation_tokens_total[1m])
rate(vllm:prompt_tokens_total[1m])

# the leading indicators
vllm:gpu_cache_usage_perc
vllm:num_requests_waiting
rate(vllm:num_preemptions_total[5m])                       # should be 0

# what users feel
histogram_quantile(0.95, rate(vllm:time_to_first_token_seconds_bucket[5m]))
histogram_quantile(0.95, rate(vllm:time_per_output_token_seconds_bucket[5m]))

# is prefix caching earning its keep?
rate(vllm:prefix_cache_hits_total[5m]) / rate(vllm:prefix_cache_queries_total[5m])

# speculation acceptance (nb 24) - below ~0.6 it may be costing you
rate(vllm:spec_decode_num_accepted_tokens_total[5m])
  / rate(vllm:spec_decode_num_draft_tokens_total[5m])
```

**Starter alert thresholds** (tune to your SLO — these are conservative defaults):

| Alert | Condition | Why |
|---|---|---|
| KV pressure | `gpu_cache_usage_perc > 0.90` for 5m | leading indicator of preemption |
| Queue depth | `num_requests_waiting > 0.3 × num_requests_running` for 5m | latency is being pushed onto users |
| Preemption | `rate(num_preemptions_total[5m]) > 0` for 10m | wasted compute, always fixable |
| TTFT SLO | `histogram_quantile(0.95, …) > your_target` | user-visible; page on this |
| Prefix regression | hit rate drops >50% week-over-week | someone reordered a prompt template |

**Per-request logging.** `--disable-log-requests` is often set in production for noise/PII reasons;
if you disable it, make sure your *gateway* logs `request_id`, prompt/completion token counts,
finish reason, and end-to-end latency — otherwise you lose the ability to attribute a bad p99 to a
particular caller. Also useful: `vllm:request_success_total{finished_reason="length"}` climbing
means clients are hitting `max_tokens` — often a prompt bug, not a capacity problem.

### Optional: generate these logs yourself (needs a T4)

The cell below launches a real server, drives load at it, and points the parsers from Parts 2–4 at
the actual output — the same code, real data.

In [ ]:
# GPU-ONLY. On Colab: Runtime > Change runtime type > T4. Skip if you're on CPU.
import torch
if not torch.cuda.is_available():
    print("No GPU detected - skipping the live-server section (everything above already ran).")
else:
    import subprocess, urllib.request, time, os
    os.environ["VLLM_LOGGING_LEVEL"] = "INFO"
    MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
    server = subprocess.Popen(
        ["vllm", "serve", MODEL, "--dtype", "half", "--max-model-len", "2048",
         "--gpu-memory-utilization", "0.85", "--max-num-seqs", "32", "--port", "8000"],
        stdout=open("vllm_live.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(180):
        try:
            urllib.request.urlopen("http://localhost:8000/health", timeout=2); break
        except Exception: time.sleep(2)

    # 1) parse the REAL startup log with the Part 1 code
    live_log = open("vllm_live.log").read()
    print("=== startup signals from the live server ===")
    for k, v in parse_startup(live_log).items():
        print(f"  {k:<18}{v}")

    # 2) drive concurrent load so the periodic stats lines have something to say
    from concurrent.futures import ThreadPoolExecutor
    from openai import OpenAI
    client = OpenAI(base_url="http://localhost:8000/v1", api_key="x")
    SYSTEM = "You are a concise assistant for a maritime museum. " * 20   # shared prefix on purpose
    def ask(i):
        return client.chat.completions.create(
            model=MODEL, max_tokens=96, temperature=0.8,
            messages=[{"role": "system", "content": SYSTEM},
                      {"role": "user", "content": f"Give three facts about ship number {i}."}])
    with ThreadPoolExecutor(max_workers=24) as ex:
        list(ex.map(ask, range(48)))

    # 3) run the Part 2 parser over the real periodic lines
    print("\n=== periodic stats lines (parsed) ===")
    for line in open("vllm_live.log"):
        p = parse_stats_line(line)
        if p: print(" ", json.dumps(p))

    # 4) scrape real /metrics and reuse Parts 3-4 verbatim
    real = parse_prometheus(urllib.request.urlopen("http://localhost:8000/metrics").read().decode())
    ttft_b = histogram_buckets(real, "vllm:time_to_first_token_seconds")
    print("\n=== real percentiles ===")
    for q in (0.5, 0.95, 0.99):
        v = histogram_quantile(q, ttft_b)
        print(f"  TTFT p{int(q*100)}: {v*1000:.0f}ms" if v == v else "  (no data)")
    print("\n=== derived signals ===")
    print(" ", json.dumps({k: round(v, 4) for k, v in derived_signals(real).items()}))

    server.terminate(); server.wait(timeout=20); print("\nserver stopped")

## Recap — what to take to production

1. **Read the startup log once per deploy.** `Available KV cache memory` and `Maximum concurrency`
   tell you your ceiling before a single user arrives. Most capacity surprises are visible here.
2. **The periodic line is your heartbeat.** Running / Waiting / KV% / prefix-hit, in that order.
3. **Percentiles come from histogram buckets** by linear interpolation — the mean will hide your
   worst 5% of users, every time.
4. **Alert on causes, not symptoms**: KV usage and queue depth lead; TTFT lags.
5. **Preemptions > 0 is always a bug in your configuration**, not a fact of life.
6. **Parse tolerantly.** Log text changes between versions; `/metrics` names are the stable contract.

### Further reading
- [vLLM production metrics docs](https://docs.vllm.ai/en/latest/usage/metrics.html) · [V1 metrics design](https://docs.vllm.ai/en/latest/design/v1/metrics.html)
- [Prometheus histograms & `histogram_quantile`](https://prometheus.io/docs/practices/histograms/) — read this before trusting any p99
- [Google SRE Book — Monitoring Distributed Systems](https://sre.google/sre-book/monitoring-distributed-systems/) (the "four golden signals")
- The mechanisms behind every metric here: notebooks [21](./Serving_Fundamentals_KV_Cache_Batching.ipynb)–[25](./Serving_Internals_Visualized_D3.ipynb)

▶ **Next:** [Serving Benchmark & Capacity Planning](./Serving_Benchmark_Capacity_Planning.ipynb) — turn
these signals into a load test, an SLO, and a cost-per-million-tokens number.